# cuDF - GPU 加速的 DataFrame 資料處理

> cuDF 提供與 pandas 幾乎相同的 API，讓你的 DataFrame 操作在 GPU 上獲得 30-150 倍加速。
> 這是資料分析師最實用的 GPU 加速工具。

## GPU 加速系列 Notebooks

1. [Numba CUDA 基礎](./01-Introduction-to-cuda-python-with-numba.ipynb)
2. [GPU 加速概覽與環境建置](./02-GPU-Acceleration-Overview-and-Environment-Setup.ipynb)
3. [CuPy - GPU 版 NumPy](./03-CuPy-GPU-Accelerated-NumPy.ipynb)
4. **[cuDF - GPU 版 pandas](./04-cuDF-GPU-Accelerated-DataFrames.ipynb) ← 目前位置**
5. [cuML - GPU 版 scikit-learn](./05-cuML-GPU-Accelerated-Machine-Learning.ipynb)
6. [cuGraph - GPU 版 NetworkX](./06-cuGraph-GPU-Accelerated-Graph-Analytics.ipynb)
7. [Dask-CUDA 多 GPU 處理](./07-Dask-CUDA-Multi-GPU-and-Large-Scale-Processing.ipynb)
8. [RAPIDS 端到端實戰](./08-RAPIDS-End-to-End-Data-Analysis-Pipeline.ipynb)

# 環境檢查
import shutil
import subprocess
import platform

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
else:
    print('未偵測到 NVIDIA GPU。')
    if platform.system() == 'Darwin' and platform.machine() == 'arm64':
        print('Apple Silicon 不支援 CUDA/RAPIDS。')
        print('Mac 替代方案: pip install polars  # 比 pandas 快 5-20x，無需 GPU')
    print('建議使用 Google Colab (免費 T4 GPU): https://colab.research.google.com')
    GPU_AVAILABLE = False

In [ ]:
# 環境檢查
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
else:
    print('未偵測到 NVIDIA GPU。')
    print('建議使用 Google Colab (免費 T4 GPU): https://colab.research.google.com')
    GPU_AVAILABLE = False

In [ ]:
# 安裝 cuDF
# !pip install cudf-cu12    # CUDA 12
# Google Colab 已預裝 cuDF

In [ ]:
import pandas as pd
import numpy as np
import time

if GPU_AVAILABLE:
    import cudf
    print(f'cuDF 版本: {cudf.__version__}')
else:
    print('本 Notebook 將只示範 pandas（CPU）版本。')

print(f'pandas 版本: {pd.__version__}')

---
## 1. cuDF vs pandas API 對照

cuDF 的設計理念：**與 pandas 語法幾乎完全相同**。

| pandas | cuDF | 說明 |
|--------|------|------|
| `pd.DataFrame({'a': [1,2]})` | `cudf.DataFrame({'a': [1,2]})` | 建立 DataFrame |
| `df.head()` | `df.head()` | 前幾筆 |
| `df.groupby('col').sum()` | `df.groupby('col').sum()` | 分組加總 |
| `df.merge(df2, on='key')` | `df.merge(df2, on='key')` | 合併 |
| `df.sort_values('col')` | `df.sort_values('col')` | 排序 |
| `df['col'].str.upper()` | `df['col'].str.upper()` | 字串操作 |
| `pd.read_csv('file.csv')` | `cudf.read_csv('file.csv')` | 讀 CSV |
| `pd.read_parquet('file.parquet')` | `cudf.read_parquet('file.parquet')` | 讀 Parquet |

> **差異極小：** 大部分情況下只需把 `pd` 換成 `cudf`，程式碼就能在 GPU 上執行。

---
## 2. 基本操作示範

In [ ]:
# pandas 版本
pdf = pd.DataFrame({
    '姓名': ['小明', '小華', '小美', '小強', '小芳'],
    '年齡': [25, 30, 28, 35, 22],
    '薪資': [45000, 52000, 48000, 65000, 38000],
    '部門': ['工程', '行銷', '工程', '管理', '行銷']
})
print('pandas DataFrame:')
print(pdf)
print(f'型別: {type(pdf)}')

In [ ]:
if GPU_AVAILABLE:
    # cuDF 版本 - 語法完全相同
    gdf = cudf.DataFrame({
        '姓名': ['小明', '小華', '小美', '小強', '小芳'],
        '年齡': [25, 30, 28, 35, 22],
        '薪資': [45000, 52000, 48000, 65000, 38000],
        '部門': ['工程', '行銷', '工程', '管理', '行銷']
    })
    print('cuDF DataFrame (GPU):')
    print(gdf)
    print(f'型別: {type(gdf)}')
    print()

    # 基本操作：與 pandas 語法相同
    print('篩選年齡 > 25:')
    print(gdf[gdf['年齡'] > 25])
    print()
    print('部門分組平均薪資:')
    print(gdf.groupby('部門')['薪資'].mean())

In [ ]:
if GPU_AVAILABLE:
    # cuDF ↔ pandas 互轉
    # cuDF → pandas (GPU → CPU)
    back_to_pandas = gdf.to_pandas()
    print(f'cuDF → pandas: {type(back_to_pandas)}')

    # pandas → cuDF (CPU → GPU)
    back_to_cudf = cudf.from_pandas(pdf)
    print(f'pandas → cuDF: {type(back_to_cudf)}')

---
## 3. 大資料量效能比較

接下來使用 **1000 萬筆** 模擬電商交易資料來比較效能差異。

In [ ]:
# 產生合成資料
N = 10_000_000  # 一千萬筆（VRAM 不足可降為 1_000_000）

np.random.seed(42)

categories = ['電子產品', '服飾', '食品', '家居', '美妝', '運動', '書籍', '玩具']
cities = ['台北', '新北', '桃園', '台中', '台南', '高雄', '新竹', '彰化']

data = {
    '訂單編號': np.arange(1, N + 1),
    '商品類別': np.random.choice(categories, N),
    '城市': np.random.choice(cities, N),
    '金額': np.random.randint(100, 50000, N).astype(np.int32),
    '數量': np.random.randint(1, 10, N).astype(np.int32),
    '折扣率': np.random.uniform(0.5, 1.0, N).astype(np.float32),
}

# pandas DataFrame
pdf = pd.DataFrame(data)
print(f'資料量: {N:,} 筆')
print(f'記憶體: {pdf.memory_usage(deep=True).sum() / 1e6:.1f} MB')
pdf.head()

In [ ]:
if GPU_AVAILABLE:
    # 傳入 GPU
    gdf = cudf.from_pandas(pdf)
    print(f'cuDF DataFrame 已載入 GPU')
    print(f'型別: {type(gdf)}')
    gdf.head()

### 3.1 GroupBy 分組聚合

In [ ]:
# CPU (pandas)
start = time.time()
result_cpu = pdf.groupby(['商品類別', '城市']).agg({
    '金額': ['sum', 'mean', 'count'],
    '數量': 'sum',
    '折扣率': 'mean'
})
cpu_time = time.time() - start
print(f'pandas groupby: {cpu_time:.4f} 秒')

# GPU (cuDF)
if GPU_AVAILABLE:
    # 暖機
    _ = gdf.groupby(['商品類別', '城市']).agg({'金額': 'sum'})

    start = time.time()
    result_gpu = gdf.groupby(['商品類別', '城市']).agg({
        '金額': ['sum', 'mean', 'count'],
        '數量': 'sum',
        '折扣率': 'mean'
    })
    gpu_time = time.time() - start
    print(f'cuDF   groupby: {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

### 3.2 Merge 合併

In [ ]:
# 建立第二個 DataFrame（城市資訊）
city_info = pd.DataFrame({
    '城市': cities,
    '人口_萬': [260, 400, 227, 282, 188, 277, 45, 127],
    '區域': ['北部', '北部', '北部', '中部', '南部', '南部', '北部', '中部']
})

# CPU (pandas)
start = time.time()
merged_cpu = pdf.merge(city_info, on='城市', how='left')
cpu_time = time.time() - start
print(f'pandas merge: {cpu_time:.4f} 秒')

# GPU (cuDF)
if GPU_AVAILABLE:
    city_info_gpu = cudf.from_pandas(city_info)

    _ = gdf.merge(city_info_gpu, on='城市', how='left')  # 暖機

    start = time.time()
    merged_gpu = gdf.merge(city_info_gpu, on='城市', how='left')
    gpu_time = time.time() - start
    print(f'cuDF   merge: {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

### 3.3 Sort 排序

In [ ]:
# CPU (pandas)
start = time.time()
sorted_cpu = pdf.sort_values(['商品類別', '金額'], ascending=[True, False])
cpu_time = time.time() - start
print(f'pandas sort: {cpu_time:.4f} 秒')

# GPU (cuDF)
if GPU_AVAILABLE:
    _ = gdf.sort_values('金額')  # 暖機

    start = time.time()
    sorted_gpu = gdf.sort_values(['商品類別', '金額'], ascending=[True, False])
    gpu_time = time.time() - start
    print(f'cuDF   sort: {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

### 3.4 篩選與計算

In [ ]:
# 複合條件篩選 + 新欄位計算

# CPU (pandas)
start = time.time()
filtered = pdf[(pdf['金額'] > 10000) & (pdf['折扣率'] < 0.8)].copy()
filtered['實付金額'] = filtered['金額'] * filtered['折扣率'] * filtered['數量']
filtered['金額等級'] = pd.cut(filtered['金額'], bins=[0, 5000, 20000, 50000],
                             labels=['低', '中', '高'])
cpu_time = time.time() - start
print(f'pandas 篩選+計算: {cpu_time:.4f} 秒 ({len(filtered):,} 筆)')

# GPU (cuDF)
if GPU_AVAILABLE:
    start = time.time()
    filtered_gpu = gdf[(gdf['金額'] > 10000) & (gdf['折扣率'] < 0.8)].copy()
    filtered_gpu['實付金額'] = filtered_gpu['金額'] * filtered_gpu['折扣率'] * filtered_gpu['數量']
    # cuDF 也支援 cut
    filtered_gpu['金額等級'] = cudf.cut(filtered_gpu['金額'], bins=[0, 5000, 20000, 50000],
                                       labels=['低', '中', '高'])
    gpu_time = time.time() - start
    print(f'cuDF   篩選+計算: {gpu_time:.4f} 秒 ({len(filtered_gpu):,} 筆)')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

---
## 4. 字串操作

In [ ]:
# 產生含字串的測試資料
N_str = 5_000_000
emails = [f'user{i}@example.com' for i in range(N_str)]

pdf_str = pd.DataFrame({'email': emails})

# CPU 字串操作
start = time.time()
pdf_str['domain'] = pdf_str['email'].str.split('@').str[1]
pdf_str['username'] = pdf_str['email'].str.split('@').str[0]
pdf_str['is_example'] = pdf_str['domain'].str.contains('example')
cpu_time = time.time() - start
print(f'pandas 字串操作: {cpu_time:.4f} 秒')

# GPU 字串操作
if GPU_AVAILABLE:
    gdf_str = cudf.DataFrame({'email': emails})

    start = time.time()
    gdf_str['domain'] = gdf_str['email'].str.split('@').str[1]
    gdf_str['username'] = gdf_str['email'].str.split('@').str[0]
    gdf_str['is_example'] = gdf_str['domain'].str.contains('example')
    gpu_time = time.time() - start
    print(f'cuDF   字串操作: {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

---
## 5. Zero-Code-Change 加速模式

RAPIDS 25.02+ 推出了 **`cudf.pandas`** 加速器，讓你 **不改任何程式碼**，既有的 pandas 程式就能自動在 GPU 上執行。

### 使用方式

在 Notebook 的 **第一個 cell** 加入：

```python
%load_ext cudf.pandas
```

之後所有 `import pandas as pd` 的操作都會自動用 GPU。

### 運作原理

```
你的程式碼: df = pd.read_csv('data.csv')
                     ↓
cudf.pandas 攔截: 嘗試用 cuDF (GPU) 執行
                     ↓
        ┌─ 支援的操作 → GPU 執行 (快！)
        └─ 不支援的操作 → 自動 fallback 到 CPU pandas
```

### 優勢
- 零學習成本：不需要學 cuDF API
- 100% 相容：不支援的操作自動回退到 pandas
- 漸進式加速：即使只有部分操作被加速也能受益

In [ ]:
# Zero-Code-Change 範例
# 注意：在實際使用時，%load_ext cudf.pandas 應放在 notebook 最頂部

if GPU_AVAILABLE:
    print('在新的 Notebook 中，只需在第一個 cell 加入：')
    print()
    print('  %load_ext cudf.pandas')
    print()
    print('之後所有 pandas 操作自動加速！例如：')
    print()
    print('  import pandas as pd  # 自動被 cudf.pandas 攔截')
    print('  df = pd.read_csv("big_data.csv")  # GPU 加速讀取')
    print('  result = df.groupby("col").sum()   # GPU 加速計算')
    print()
    print('不支援的操作會自動回退到 CPU，不會出錯。')
else:
    print('cudf.pandas 需要 GPU 環境。')

---
## 6. 讀寫檔案 (GPU 加速 I/O)

In [ ]:
import tempfile
import os

# 產生測試 CSV
N_io = 2_000_000
tmp_csv = os.path.join(tempfile.gettempdir(), 'test_rapids.csv')

df_io = pd.DataFrame({
    'id': range(N_io),
    'value_a': np.random.rand(N_io).astype(np.float32),
    'value_b': np.random.rand(N_io).astype(np.float32),
    'category': np.random.choice(['A', 'B', 'C', 'D'], N_io)
})
df_io.to_csv(tmp_csv, index=False)
file_size = os.path.getsize(tmp_csv) / 1e6
print(f'測試檔案: {tmp_csv} ({file_size:.1f} MB)')

# CPU 讀取
start = time.time()
df_cpu = pd.read_csv(tmp_csv)
cpu_time = time.time() - start
print(f'pandas read_csv: {cpu_time:.4f} 秒')

# GPU 讀取
if GPU_AVAILABLE:
    start = time.time()
    df_gpu = cudf.read_csv(tmp_csv)
    gpu_time = time.time() - start
    print(f'cuDF   read_csv: {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

# 清理暫存檔
os.remove(tmp_csv)

### Parquet 格式更快

在 GPU 工作流中，建議使用 **Parquet** 格式而非 CSV：
- 二進位格式，讀寫更快
- 支援列式儲存，只讀需要的欄位
- 自帶壓縮，檔案更小
- cuDF 對 Parquet 的加速效果比 CSV 更好

```python
# 寫入 Parquet
gdf.to_parquet('data.parquet')

# 讀取 Parquet（只讀需要的欄位）
gdf = cudf.read_parquet('data.parquet', columns=['col1', 'col2'])
```

---
## 7. Polars GPU Engine 簡介

[Polars](https://pola.rs/) 是近年崛起的高效能 DataFrame 套件，自 2025 年起整合了 RAPIDS cuDF 作為 GPU 後端。

```python
import polars as pl

# 啟用 GPU 引擎
result = (
    pl.scan_parquet('large_data.parquet')
    .filter(pl.col('amount') > 1000)
    .group_by('category')
    .agg(pl.col('amount').sum())
    .collect(engine='gpu')  # 使用 GPU 執行
)
```

### Polars GPU vs cuDF

| 特性 | cuDF | Polars GPU |
|------|------|------------|
| API 風格 | pandas-like | Polars 原生 (lazy evaluation) |
| 成熟度 | 高 (2018 年起) | 新 (2025 年) |
| CPU fallback | 手動處理 | 自動 |
| 超出 VRAM | 需 Dask-CUDA | 支援 out-of-core |

> **建議：** 如果你已經在用 pandas → 選 cuDF。如果你在用 Polars → 試試 GPU engine。

---
## 8. cuDF 使用注意事項

### 與 pandas 的差異

- **Index**：cuDF 不完全支援 MultiIndex 的所有操作
- **apply()**：cuDF 的 `apply()` 功能有限，建議用向量化操作取代
- **資料型別**：cuDF 不支援 `object` dtype（需轉為 `string`）
- **排序穩定性**：GPU 排序預設不穩定（相同值的相對順序可能改變）

### 效能最佳化建議

1. **避免 `apply()`**：改用向量化操作（直接欄位運算）
2. **使用 Parquet**：比 CSV 讀寫快數倍
3. **批次處理**：一次處理大量資料，而非逐筆處理
4. **減少 CPU-GPU 傳輸**：盡可能全程使用 cuDF
5. **注意 VRAM**：`del df` 釋放不需要的 DataFrame

---
## 9. 效能總結

| 操作 | 典型加速倍數 | 說明 |
|------|-------------|------|
| `groupby().agg()` | 30-100x | 分組愈多加速愈明顯 |
| `merge()` | 20-80x | 大表合併效果顯著 |
| `sort_values()` | 30-100x | GPU 平行排序 |
| `read_csv()` | 5-20x | I/O 瓶頸限制上限 |
| `read_parquet()` | 10-30x | 比 CSV 加速更多 |
| 篩選 + 計算 | 20-60x | 向量化操作 |
| 字串操作 | 10-40x | GPU 字串支援持續改善 |

### 經驗法則

- 資料量 < 10 萬筆 → 用 pandas 就好
- 資料量 10-100 萬筆 → cuDF 開始有優勢
- 資料量 > 100 萬筆 → cuDF 明顯更快
- 資料量 > GPU VRAM → 需搭配 Dask-CUDA

---
## 參考資源

- [cuDF 官方文件](https://docs.rapids.ai/api/cudf/stable/)
- [cuDF 與 pandas 差異](https://docs.rapids.ai/api/cudf/stable/cudf_pandas/)
- [cudf.pandas 加速器](https://docs.rapids.ai/api/cudf/stable/cudf_pandas/)
- [Polars GPU Engine](https://docs.pola.rs/user-guide/gpu-support/)

> **下一步：** 前往 [05-cuML-GPU-Accelerated-Machine-Learning.ipynb](./05-cuML-GPU-Accelerated-Machine-Learning.ipynb) 學習 GPU 加速的機器學習。